In [1]:
from collections import deque
import heapq
import time


def take_maze():
    rows = int(input("Rows: "))
    cols = int(input("Columns: "))

    maze = []

    print("Enter 0 for path and 1 for wall")

    for i in range(rows):
        while True:
            row = input("Row " + str(i + 1) + ": ")
            row = row.replace(" ", "")

            if len(row) == cols and all(x in "01" for x in row):
                maze.append([int(x) for x in row])
                break

            print("Enter", cols, "digits using only 0 and 1")

    print("Enter start position")
    start_row = int(input("Start row: "))
    start_col = int(input("Start column: "))

    print("Enter goal position")
    goal_row = int(input("Goal row: "))
    goal_col = int(input("Goal column: "))

    start = (start_row, start_col)
    goal = (goal_row, goal_col)

    if not (0 <= start_row < rows and 0 <= start_col < cols):
        raise ValueError("Invalid start position")

    if not (0 <= goal_row < rows and 0 <= goal_col < cols):
        raise ValueError("Invalid goal position")

    if maze[start_row][start_col] == 1:
        raise ValueError("Start cannot be a wall")

    if maze[goal_row][goal_col] == 1:
        raise ValueError("Goal cannot be a wall")

    return maze, start, goal


def neighbours(position, maze):
    row, col = position

    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    result = []

    for dr, dc in moves:
        new_row = row + dr
        new_col = col + dc

        if (0 <= new_row < len(maze) and
            0 <= new_col < len(maze[0])):

            if maze[new_row][new_col] == 0:
                result.append((new_row, new_col))

    return result


def make_path(parent, start, goal):
    if goal not in parent:
        return []

    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


def bfs(maze, start, goal):
    start_time = time.perf_counter()

    queue = deque([start])
    visited = {start}
    parent = {start: None}

    generated = 1
    expanded = 0
    max_frontier = 1

    while queue:
        current = queue.popleft()
        expanded += 1

        if current == goal:
            break

        for next_position in neighbours(current, maze):
            if next_position not in visited:
                visited.add(next_position)
                parent[next_position] = current
                queue.append(next_position)
                generated += 1

        max_frontier = max(max_frontier, len(queue))

    path = make_path(parent, start, goal)
    elapsed = (time.perf_counter() - start_time) * 1000

    return {
        "Algorithm": "BFS",
        "Path": path,
        "Path Length": len(path) - 1 if path else "No path",
        "Generated": generated,
        "Expanded": expanded,
        "Frontier": max_frontier,
        "Time": round(elapsed, 6)
    }


def dfs(maze, start, goal):
    start_time = time.perf_counter()

    stack = [start]
    visited = {start}
    parent = {start: None}

    generated = 1
    expanded = 0
    max_frontier = 1

    while stack:
        current = stack.pop()
        expanded += 1

        if current == goal:
            break

        for next_position in neighbours(current, maze):
            if next_position not in visited:
                visited.add(next_position)
                parent[next_position] = current
                stack.append(next_position)
                generated += 1

        max_frontier = max(max_frontier, len(stack))

    path = make_path(parent, start, goal)
    elapsed = (time.perf_counter() - start_time) * 1000

    return {
        "Algorithm": "DFS",
        "Path": path,
        "Path Length": len(path) - 1 if path else "No path",
        "Generated": generated,
        "Expanded": expanded,
        "Frontier": max_frontier,
        "Time": round(elapsed, 6)
    }


def heuristic(position, goal):
    return abs(position[0] - goal[0]) + abs(position[1] - goal[1])


def astar(maze, start, goal):
    start_time = time.perf_counter()

    priority_queue = []
    counter = 0

    start_cost = heuristic(start, goal)
    heapq.heappush(priority_queue, (start_cost, counter, start))

    parent = {start: None}
    cost = {start: 0}
    visited = set()

    generated = 1
    expanded = 0
    max_frontier = 1

    while priority_queue:
        f_score, old_counter, current = heapq.heappop(priority_queue)

        if current in visited:
            continue

        visited.add(current)
        expanded += 1

        if current == goal:
            break

        for next_position in neighbours(current, maze):
            if next_position in visited:
                continue

            new_cost = cost[current] + 1

            if next_position not in cost or new_cost < cost[next_position]:
                cost[next_position] = new_cost
                parent[next_position] = current

                h_score = heuristic(next_position, goal)
                total_cost = new_cost + h_score

                counter += 1

                heapq.heappush(
                    priority_queue,
                    (total_cost, counter, next_position)
                )

                generated += 1

        max_frontier = max(max_frontier, len(priority_queue))

    path = make_path(parent, start, goal)
    elapsed = (time.perf_counter() - start_time) * 1000

    return {
        "Algorithm": "A*",
        "Path": path,
        "Path Length": len(path) - 1 if path else "No path",
        "Generated": generated,
        "Expanded": expanded,
        "Frontier": max_frontier,
        "Time": round(elapsed, 6)
    }

In [2]:
def show_result(result):
    print()
    print(result["Algorithm"])
    print("Path:", result["Path"])
    print("Path length:", result["Path Length"])
    print("Nodes generated:", result["Generated"])
    print("Nodes expanded:", result["Expanded"])
    print("Maximum frontier:", result["Frontier"])
    print("Time:", result["Time"], "ms")


def show_table(results):
    print()
    print("Comparison")

    print("{:<12} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
        "Algorithm", "Path", "Generated", "Expanded", "Frontier", "Time"
    ))

    print("-" * 72)

    for result in results:
        print("{:<12} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
            result["Algorithm"],
            result["Path Length"],
            result["Generated"],
            result["Expanded"],
            result["Frontier"],
            result["Time"]
        ))


def run_maze(number):
    print()
    print("Maze", number)

    maze, start, goal = take_maze()

    results = [
        bfs(maze, start, goal),
        dfs(maze, start, goal),
        astar(maze, start, goal)
    ]

    for result in results:
        show_result(result)

    show_table(results)
    return results

In [3]:
print("Maze Pathfinding using BFS, DFS and A*")

number_of_mazes = int(input("How many mazes: "))

if number_of_mazes <= 0:
    raise ValueError("Enter at least one maze")

all_results = []

for i in range(1, number_of_mazes + 1):
    results = run_maze(i)

    for result in results:
        all_results.append({
            "Maze": i,
            "Algorithm": result["Algorithm"],
            "Path Length": result["Path Length"],
            "Generated": result["Generated"],
            "Expanded": result["Expanded"],
            "Frontier": result["Frontier"],
            "Time": result["Time"]
        })

print()
print("Final results")

print("{:<8} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
    "Maze", "Algorithm", "Path", "Generated",
    "Expanded", "Frontier", "Time"
))

print("-" * 88)

for result in all_results:
    print("{:<8} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
        result["Maze"],
        result["Algorithm"],
        result["Path Length"],
        result["Generated"],
        result["Expanded"],
        result["Frontier"],
        result["Time"]
    ))

Maze Pathfinding using BFS, DFS and A*
How many mazes: 1

Maze 1
Rows: 3
Columns: 3
Enter 0 for path and 1 for wall
Row 1: 000
Row 2: 010
Row 3: 000
Enter start position
Start row: 0
Start column: 0
Enter goal position
Goal row: 2
Goal column: 2

BFS
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]
Path length: 4
Nodes generated: 8
Nodes expanded: 8
Maximum frontier: 2
Time: 0.03867 ms

DFS
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
Path length: 4
Nodes generated: 6
Nodes expanded: 5
Maximum frontier: 2
Time: 0.01389 ms

A*
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]
Path length: 4
Nodes generated: 8
Nodes expanded: 8
Maximum frontier: 2
Time: 0.03388 ms

Comparison
Algorithm    Path         Generated    Expanded     Frontier     Time        
------------------------------------------------------------------------
BFS          4            8            8            2            0.03867     
DFS          4            6            5            2            0.01389     
A*           

In [4]:
def objective(x):
    return -(x - 5) ** 2 + 25


x = int(input("Enter starting value (0 to 10): "))

if x < 0 or x > 10:
    raise ValueError("Enter a value between 0 and 10.")

steps = 0

while True:
    candidates = []

    if x > 0:
        candidates.append(x - 1)

    if x < 10:
        candidates.append(x + 1)

    better = [value for value in candidates if objective(value) > objective(x)]

    if not better:
        break

    x = max(better, key=objective)
    steps += 1

print("Final value of x:", x)
print("Maximum function value:", objective(x))
print("Steps taken:", steps)

Enter starting value (0 to 10): 4
Final value of x: 5
Maximum function value: 25
Steps taken: 1


In [5]:
colors = ["Red", "Green", "Blue"]

graph = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"]
}

assignment = {}
checks = 0


def color_map(index):
    global checks

    regions = list(graph.keys())

    if index == len(regions):
        return True

    region = regions[index]

    for color in colors:
        checks += 1
        valid = True

        for neighbor in graph[region]:
            if assignment.get(neighbor) == color:
                valid = False
                break

        if valid:
            assignment[region] = color

            if color_map(index + 1):
                return True

            del assignment[region]

    return False


solved = color_map(0)

print("Map Coloring Result")

if solved:
    for region, color in assignment.items():
        print(region, ":", color)
else:
    print("No valid coloring found.")

print("Color assignment checks:", checks)

Map Coloring Result
A : Red
B : Green
C : Blue
D : Red
Color assignment checks: 7
